In [1]:
!git clone https://github.com/HenriqueSchmitz/mario-the-explorer

Cloning into 'mario-the-explorer'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 350 (delta 3), reused 11 (delta 2), pack-reused 335 (from 1)
Receiving objects: 100% (350/350), 1.34 MiB | 11.25 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [2]:
!sh ./mario-the-explorer/setup.sh

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 165.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 325.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 301.8 MB/s eta 0:00:00
Importing SuperMarioWorld-Snes-v0
Imported 1 games


In [3]:
!pip install -q stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 40.8 MB/s eta 0:00:00


In [4]:
from typing import Optional
from enum import Enum
from logging import Logger

import cv2
import torch
import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.logger import KVWriter, Logger as PpoLogger

from mario_the_explorer import (SuperMarioWorldLayeredEmulator, RewardModel, ScreenOverlay, Tile, get_file_logger,
                                tile_absolute_id, TileEncoder, SuperMarioAction, SuperMarioCombo, SuperMarioDiscretizer,
                                prime_policy_for_combo, TileType, SCREEN_COLUMNS, SCREEN_ROWS, TILE_SIZE, ButtonStates)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
class Direction(Enum):
    LEFT = 0
    RIGHT = 1
    UP = 2
    DOWN = 3

In [7]:
from re import T
from mario_the_explorer.environment import tiles
class TryThingsRewardModel(RewardModel):
    def __init__(self, new_block_reward = 10.0, action_reward = 1.0, progression_reward = 0.1):
        self._blocks_seen = set()
        self._block_action_counts = {}
        self._block_available_rewards = {}
        self._best_distance_reached = 0
        self._new_block_reward = new_block_reward
        self._action_reward = action_reward
        self._progression_reward = progression_reward
        self.MAX_BLOCK_REWARD = self._get_total_reward_for_block()

    def reset(self) -> None:
        self._blocks_seen = set()
        self._block_action_counts = {}
        self._block_available_rewards = {}
        self._best_distance_reached = 0

    def _get_total_reward_for_block(self, number_of_actions = 10, number_of_directions = 4) -> float:
        total_possible_reward = 0.0
        last_total_possible_reward = -1
        interactions = 0
        while total_possible_reward != last_total_possible_reward:
            last_total_possible_reward = total_possible_reward
            interactions += 1
            total_possible_reward += self._reward_curve(interactions)
        return total_possible_reward * number_of_actions * number_of_directions

    def _reward_curve(self, action_count: int) -> float:
        action_reward = self._action_reward / action_count
        if action_reward < 0.05:
            action_reward = 0.0
        return action_reward

    def get_reward(self,
                   action: list[int],
                   observation: list[list[Tile]],
                   terminated: bool,
                   truncated: bool,
                   info: dict) -> float:
        reward = 0.0
        for row in observation:
            for tile in row:
                tile_id = tile_absolute_id(tile)
                if tile["type"] == TileType.EMPTY:
                    self._block_available_rewards[tile_id] = 0
                    continue
                if tile["type"] == TileType.MARIO:
                    self._block_available_rewards[tile_id] = 0
                    continue
                if tile_id not in self._blocks_seen:
                    self._blocks_seen.add(tile_id)
                    reward += 10.0
                    self._block_available_rewards[tile_id] = self.MAX_BLOCK_REWARD
        tiles_around_mario = self._get_tiles_around_mario(observation)
        for tile_and_direction in tiles_around_mario:
            combo_id = SuperMarioCombo.get_combo_id_from_action(action)
            if combo_id == SuperMarioCombo.DO_NOTHING:
                continue
            interaction_id = (combo_id, tile_and_direction[0], tile_and_direction[1])
            action_reward = 0.0
            if interaction_id not in self._block_action_counts:
                self._block_action_counts[interaction_id] = 0
            self._block_action_counts[interaction_id] += 1
            action_reward = self._reward_curve(self._block_action_counts[interaction_id])
            self._block_available_rewards[tile_and_direction[1]] -= action_reward
            reward += action_reward
        if info["x"] > self._best_distance_reached:
            self._best_distance_reached = info["x"]
            reward += 0.1
        if terminated or truncated:
            unique_tiles_visible = set()
            for row in observation:
                for tile in row:
                    unique_tiles_visible.add(tile_absolute_id(tile))
            valid_tiles_visible = len(unique_tiles_visible) - 2
            if valid_tiles_visible == 0:
                valid_tiles_visible = 1
            max_reward_possible = valid_tiles_visible * self.MAX_BLOCK_REWARD
            reward_still_available = 0
            for tile_id in self._block_available_rewards:
                reward_still_available += self._block_available_rewards[tile_id]
            explored_percentage = reward_still_available / max_reward_possible
            reward -= explored_percentage * 20.0
        return reward

    def _get_tiles_around_mario(self, observation: list[list[Tile]]) -> set[tuple[Direction, int]]:
        mario_coordinates = self._find_mario_coordinates(observation)
        blocks_around_mario = set()
        if not mario_coordinates:
            return blocks_around_mario
        for mario_row, mario_col in mario_coordinates:
            if mario_row > 0:
                block_above_mario = observation[mario_row - 1][mario_col]
                if block_above_mario["type"] != TileType.MARIO and block_above_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.UP.name, tile_absolute_id(block_above_mario)))
            if mario_row < len(observation) - 1:
                block_below_mario = observation[mario_row + 1][mario_col]
                if block_below_mario["type"] != TileType.MARIO and block_below_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.DOWN.name, tile_absolute_id(block_below_mario)))
            if mario_col > 0:
                block_left_of_mario = observation[mario_row][mario_col - 1]
                if block_left_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.LEFT.name, tile_absolute_id(block_left_of_mario)))
            if mario_col < len(observation[0]) - 1:
                block_right_of_mario = observation[mario_row][mario_col + 1]
                if block_right_of_mario["type"] != TileType.EMPTY:
                    blocks_around_mario.add((Direction.RIGHT.name, tile_absolute_id(block_right_of_mario)))
        return blocks_around_mario


    def _find_mario_coordinates(self, observation: list[list[Tile]]) -> list[tuple[int, int]]:
        mario_coordinates = []
        for row_idx, row in enumerate(observation):
            for col_idx, tile in enumerate(row):
                if tile["type"] == TileType.MARIO:
                    mario_coordinates.append((row_idx, col_idx))
        return mario_coordinates

    def get_potential_map(self, observation: list[list[Tile]]) -> np.ndarray:
        h, w = len(observation), len(observation[0])
        potential_map = np.zeros((h, w), dtype=np.float32)
        for r in range(h):
            for c in range(w):
                t_id = tile_absolute_id(observation[r][c])
                potential_map[r, c] = self._block_available_rewards.get(t_id, 0.0)/self.MAX_BLOCK_REWARD
        return potential_map

In [8]:
class MarioReshapeWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        c, h, w = self.observation_space.shape
        self.observation_space = gym.spaces.Box(
            low=0,
            high=255,
            shape=(c+1, h, w),
            dtype=np.float32
        )

    def observation(self, obs):
        reward_model = self.env.unwrapped.reward_model
        heatmap = reward_model.get_potential_map(self.env.unwrapped.observation)
        heatmap = np.expand_dims(heatmap, axis=0)
        screen = np.concat([obs, heatmap], axis=0).astype(np.float32)
        return screen

In [9]:
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
import torch.nn as nn

class CustomMarioCNN(BaseFeaturesExtractor):
    def __init__(self, observation_space: gym.spaces.Box, features_dim: int = 512, action_space: int = 11):
        super().__init__(observation_space, features_dim)
        self._init_params = {
            "observation_space": observation_space,
            "features_dim": features_dim,
            "action_space": action_space,
        }
        n_input_channels = observation_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels, 32, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=1),
            nn.ReLU(),
            nn.Flatten()
        )

        # Compute shape by doing one forward pass
        with torch.no_grad():
            sample_tensor = torch.as_tensor(observation_space.sample()[None]).float()
            n_flatten = self.cnn(sample_tensor).shape[1]

        self.linear = nn.Sequential(
            nn.Linear(n_flatten, features_dim),
            nn.ReLU(),
            nn.Linear(features_dim, features_dim),
            nn.ReLU()
        )

        self.output = nn.Sequential(
            nn.Linear(features_dim, action_space),
            nn.Softmax(dim=-1)
        )

    def forward(self, observations: torch.Tensor, deterministic=False) -> torch.Tensor:
        logits = self.linear(self.cnn(observations))
        probs = self.output(logits)
        if deterministic:
            return probs.argmax(dim=-1)
        return torch.multinomial(probs, num_samples=1, replacement=True)

    def save(self, path: str):
        checkpoint = {
            "state_dict": self.state_dict(),
            "params": self._init_params
        }
        torch.save(checkpoint, path)

    @classmethod
    def load(cls, path: str):
        checkpoint = torch.load(path, weights_only=False)
        model = cls(**checkpoint["params"])
        model.load_state_dict(checkpoint["state_dict"])
        model.eval()
        return model

def get_weights(model):
    """Extracts all weights of the model into a single flat array."""
    return np.concatenate([p.data.cpu().numpy().flatten() for p in model.parameters()])

def set_weights(model, weights):
    """Sets the model weights from a flat array."""
    start = 0
    for p in model.parameters():
        shape = p.data.shape
        size = np.prod(shape)
        new_data = weights[start:start + size].reshape(shape)
        p.data.copy_(torch.from_numpy(new_data))
        start += size

In [10]:
GRID_COLOR_RGB = (80, 80, 80)

class RewardHeatmapOverlay(ScreenOverlay):
    def __init__(self, reward_model: TryThingsRewardModel):
        self._reward_model = reward_model
        self.img_width = SCREEN_COLUMNS * TILE_SIZE
        self.img_height = SCREEN_ROWS * TILE_SIZE

    def apply(self, original_frame: np.ndarray, observation: Optional[list[list[Tile]]]):
        if observation is None:
            return original_frame
        reward_map = self._reward_model.get_potential_map(observation)
        reward_image = self._get_reward_image(reward_map)
        return np.vstack((original_frame, reward_image))

    def _get_reward_image(self, reward_map):
        matrix_img = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        matrix_img = self._populate_objects(matrix_img, reward_map)
        matrix_img = self._draw_grid(matrix_img)
        unused_region = np.zeros((self.img_height, self.img_width, 3), dtype=np.uint8)
        return np.hstack((matrix_img, unused_region))

    def _populate_objects(self, matrix_img, reward_map):
        for row in range(SCREEN_ROWS):
            for col in range(SCREEN_COLUMNS):
                tile_available_reward = reward_map[row][col]
                self._draw_tile(matrix_img, col, row, tile_available_reward)
        return matrix_img

    def _draw_tile(self, img, x: int, y: int, tile_available_reward: float) -> None:
        scaled_reward = int(tile_available_reward * 255)
        visually_uniform_grayscale = scaled_reward ** 2.2
        color = (visually_uniform_grayscale, visually_uniform_grayscale, visually_uniform_grayscale)
        x_start = int(x * TILE_SIZE)
        y_start = int(y * TILE_SIZE)
        x_end = x_start + TILE_SIZE - 1
        y_end = y_start + TILE_SIZE - 1
        cv2.rectangle(img, (x_start, y_start), (x_end, y_end), color, -1)

    def _draw_grid(self, img):
        h, w = img.shape[:2]
        for x in range(0, w + 1, TILE_SIZE):
            cv2.line(img, (x, 0), (x, h), GRID_COLOR_RGB, 1)
        for y in range(0, h + 1, TILE_SIZE):
            cv2.line(img, (0, y), (w, y), GRID_COLOR_RGB, 1)
        return img

In [11]:
def run_mario_episode(model, env):
    model = model.to(device)
    obs, info = env.reset()
    done = False
    total_reward = 0

    model.eval()

    with torch.no_grad():
        while not done:
            obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).to(device)
            action = model(obs_tensor, deterministic=True).item()
            obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            total_reward += reward

    return total_reward

In [12]:
def sparse_mutate(weights, power, rate=0.1):
    gene_count = weights.shape[0]
    mask = np.random.rand(gene_count) < rate
    mutation = np.random.normal(0, power, size=np.sum(mask))
    weights[mask] += mutation
    return weights

def tournament_selection(population, fitness_scores, k=3):
    """
    Randomly picks 'k' individuals and returns the one with the highest fitness.
    This maintains diversity better than just picking the absolute top performers.
    """
    selection_indices = np.random.choice(len(population), k, replace=False)
    best_index = selection_indices[np.argmax([fitness_scores[i] for i in selection_indices])]
    return population[best_index]

def crossover(parent1_weights, parent2_weights):
    """
    Performs uniform crossover by swapping weight values between two parents.
    """
    # Create a mask to decide which parent provides which weight
    mask = np.random.rand(*parent1_weights.shape) > 0.5
    child_weights = np.where(mask, parent1_weights, parent2_weights)
    return child_weights

In [13]:
RUN_NAME = "smarter_genetic_algorithm_full_fps"
LEVEL = "DonutPlains1"
LOG_LEVEL = "INFO"
MAX_STEPS = 4000
POPULATION_SIZE = 10
ELITE_SIZE = 3
TOTAL_GENERATIONS = 10
MUTATION_POWER = 0.05
AGGRESSIVE_RATIO = 0.15
SPARSE_RATE = 0.05
CROSSOVER_RATE = 0.4

In [14]:
logger = get_file_logger(RUN_NAME, LOG_LEVEL)
reward_model = TryThingsRewardModel()
overlay = RewardHeatmapOverlay(reward_model)
base_env = SuperMarioWorldLayeredEmulator(level = LEVEL,
                                          render_mode = "rgb_array",
                                          reward_model = reward_model,
                                          screen_overlay = overlay,
                                          render_debug = True,
                                          render_grid = True,
                                          max_episode_length = MAX_STEPS,
                                          button_states = ButtonStates(yellow_button=True),
                                          logger = logger)
env = SuperMarioDiscretizer(base_env)
env = MarioReshapeWrapper(env)

2026-05-08 22:23:46 [INFO] Session log for run smarter_genetic_algorithm_full_fps with level [INFO] initialized at: smarter_genetic_algorithm_full_fps_20260508_222346.log


In [ ]:
population = [CustomMarioCNN(env.observation_space, features_dim=512) for _ in range(POPULATION_SIZE)]

for gen in range(TOTAL_GENERATIONS):
    fitness_scores = []
    for model in population:
        score = run_mario_episode(model, env)
        fitness_scores.append(score)

    elite_indices = np.argsort(fitness_scores)[-ELITE_SIZE:]
    elites = [population[i] for i in elite_indices]

    new_population = []
    new_population.extend(elites)

    while len(new_population) < POPULATION_SIZE:
        parent1 = tournament_selection(population, fitness_scores, k=3)
        parent2 = tournament_selection(population, fitness_scores, k=3)
        p1_weights = get_weights(parent1)
        p2_weights = get_weights(parent2)
        if np.random.rand() < CROSSOVER_RATE:
            child_weights = crossover(p1_weights, p2_weights)
        else:
            child_weights = p1_weights.copy()
        current_fill_count = len(new_population) - ELITE_SIZE
        total_to_fill = POPULATION_SIZE - ELITE_SIZE
        if (current_fill_count / total_to_fill) < AGGRESSIVE_RATIO:
            child_weights += np.random.normal(0, MUTATION_POWER, size=p1_weights.shape)
        else:
            mask = np.random.rand(*p1_weights.shape) < SPARSE_RATE
            noise = np.random.normal(0, MUTATION_POWER, size=np.sum(mask))
            child_weights[mask] += noise
        child_model = CustomMarioCNN(env.observation_space, features_dim=512)
        set_weights(child_model, child_weights)
        new_population.append(child_model)

    population = new_population
    logger.info(f"Gen {gen} Best Fitness: {max(fitness_scores)}")

In [ ]:
best_model = population[np.argmax(fitness_scores)]
best_model.save(f"trial_{RUN_NAME}.pth")

In [ ]:
from gymnasium.wrappers import RecordVideo

try:
    env.env.env._max_episode_length = 8000
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trial-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    best_model.eval()
    done = False
    step_count = 0
    with torch.no_grad():
        while not done:
            env.render()
            obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).to(device)
            action = best_model(obs_tensor, deterministic=True)
            obs, reward, terminated, truncated, info = video_env.step(action)
            step_count += 1
            if step_count % 1000 == 0:
                logger.info(f"Step: {step_count}")
            done = terminated or truncated
            if done:
                logger.info(f"Terminated: {terminated}")
                logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

In [ ]:
POPULATION_SIZE = 50
ELITE_SIZE = 5
TOTAL_GENERATIONS = 50
MUTATION_POWER = 0.02
MAX_STEPS = 10000

In [ ]:
env.env.env._max_episode_length = MAX_STEPS
for gen in range(TOTAL_GENERATIONS):
    fitness_scores = []
    for model in population:
        score = run_mario_episode(model, env)
        fitness_scores.append(score)

    elite_indices = np.argsort(fitness_scores)[-ELITE_SIZE:]
    elites = [population[i] for i in elite_indices]

    new_population = []
    new_population.extend(elites)

    while len(new_population) < POPULATION_SIZE:
        parent1 = tournament_selection(population, fitness_scores, k=3)
        parent2 = tournament_selection(population, fitness_scores, k=3)
        p1_weights = get_weights(parent1)
        p2_weights = get_weights(parent2)
        if np.random.rand() < CROSSOVER_RATE:
            child_weights = crossover(p1_weights, p2_weights)
        else:
            child_weights = p1_weights.copy()
        current_fill_count = len(new_population) - ELITE_SIZE
        total_to_fill = POPULATION_SIZE - ELITE_SIZE
        if (current_fill_count / total_to_fill) < AGGRESSIVE_RATIO:
            child_weights += np.random.normal(0, MUTATION_POWER, size=p1_weights.shape)
        else:
            mask = np.random.rand(*p1_weights.shape) < SPARSE_RATE
            noise = np.random.normal(0, MUTATION_POWER, size=np.sum(mask))
            child_weights[mask] += noise
        child_model = CustomMarioCNN(env.observation_space, features_dim=512)
        set_weights(child_model, child_weights)
        new_population.append(child_model)

    population = new_population
    logger.info(f"Gen {gen} Best Fitness: {max(fitness_scores)}")

In [ ]:
best_model = population[np.argmax(fitness_scores)]
best_model.save(f"trained_{RUN_NAME}.pth")

In [15]:
best_model = CustomMarioCNN.load(f"trained_{RUN_NAME}.pth")

In [16]:
from gymnasium.wrappers import RecordVideo

try:
    env.env.env._max_episode_length = 16000
    video_env = RecordVideo(env, video_folder="./", name_prefix=f"trained-{RUN_NAME}", episode_trigger=lambda x: True)
    obs, info = video_env.reset()
    best_model.eval()
    done = False
    step_count = 0
    with torch.no_grad():
        while not done:
            env.render()
            obs_tensor = torch.from_numpy(obs).float().unsqueeze(0).to(device)
            action = best_model(obs_tensor, deterministic=True)
            obs, reward, terminated, truncated, info = video_env.step(action)
            step_count += 1
            if step_count % 1000 == 0:
                logger.info(f"Step: {step_count}")
            done = terminated or truncated
            if done:
                logger.info(f"Terminated: {terminated}")
                logger.info(f"Truncated: {truncated}")
except Exception as e:
    logger.error(e)
    raise e
finally:
    video_env.close()

/usr/local/lib/python3.12/dist-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /content folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(
2026-05-08 22:23:56 [INFO] Step: 1000
2026-05-08 22:24:06 [INFO] Step: 2000
2026-05-08 22:24:07 [INFO] Terminated: True
2026-05-08 22:24:07 [INFO] Truncated: False
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
for i, elite_model in enumerate(elites):
    elite_model.save(f"elite_{i}.pth")

In [ ]:
POPULATION_SIZE = 100
ELITE_SIZE = 10
TOTAL_GENERATIONS = 100
MUTATION_POWER = 0.02
MAX_STEPS = 10000

In [ ]:
env.env.env._max_episode_length = MAX_STEPS
for gen in range(TOTAL_GENERATIONS):
    fitness_scores = []
    for model in population:
        score = run_mario_episode(model, env)
        fitness_scores.append(score)

    elite_indices = np.argsort(fitness_scores)[-ELITE_SIZE:]
    elites = [population[i] for i in elite_indices]

    new_population = []
    new_population.extend(elites)

    while len(new_population) < POPULATION_SIZE:
        parent1 = tournament_selection(population, fitness_scores, k=3)
        parent2 = tournament_selection(population, fitness_scores, k=3)
        p1_weights = get_weights(parent1)
        p2_weights = get_weights(parent2)
        if np.random.rand() < CROSSOVER_RATE:
            child_weights = crossover(p1_weights, p2_weights)
        else:
            child_weights = p1_weights.copy()
        current_fill_count = len(new_population) - ELITE_SIZE
        total_to_fill = POPULATION_SIZE - ELITE_SIZE
        if (current_fill_count / total_to_fill) < AGGRESSIVE_RATIO:
            child_weights += np.random.normal(0, MUTATION_POWER, size=p1_weights.shape)
        else:
            mask = np.random.rand(*p1_weights.shape) < SPARSE_RATE
            noise = np.random.normal(0, MUTATION_POWER, size=np.sum(mask))
            child_weights[mask] += noise
        child_model = CustomMarioCNN(env.observation_space, features_dim=512)
        set_weights(child_model, child_weights)
        new_population.append(child_model)

    population = new_population
    logger.info(f"Gen {gen} Best Fitness: {max(fitness_scores)}")